In [0]:
df = spark.read.csv(
    '/Volumes/workspace/default/dataset_olist/olist_cleaned/',
    header=True, inferSchema=True
)

# Re-import needed functions
from pyspark.sql.functions import col, count, avg, desc
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import round as spark_round

print(f'Loaded: {df.count():,} rows')
df.printSchema()

In [0]:
# CREATE BINARY LABEL AND PREPARE FEATURES

from pyspark.sql.functions import col, when

# review_score 1,2,3 = Negative = label 0
# review_score 4,5   = Positive = label 1
ml_df = df.withColumn(
    'label',
    when(col('review_score') >= 4, 1).otherwise(0)
)

# Keep only numeric feature columns and label
feature_cols = ['delivery_days', 'total_price', 'freight_value',
                'payment_installments', 'item_count']

ml_df = ml_df.select(feature_cols + ['label']).dropna()

print(f'ML dataset size: {ml_df.count():,} rows')
print()
print('Label distribution (0=Negative review, 1=Positive review):')
ml_df.groupBy('label').count().orderBy('label').show()

In [0]:
# SPLIT DATA: 80% TRAIN (FOR GRID SEARCH CV) / 20% TEST

# CrossValidator performs its own internal k-fold validation on train_df,so a separate validation set is not needed here.
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f'Training samples (used by Grid Search CV) : {train_df.count():,}')
print(f'Test     samples (final evaluation only)  : {test_df.count():,}')
print()
print('IMPORTANT: test_df is locked until the very end.')
print('Grid Search finds the best params using cross-validation on train_df only.')

In [0]:
# BUILD PIPELINE, PARAM GRID, AND CROSS VALIDATOR

import os
# Set temp path for Spark ML on serverless cluster (must be set before CrossValidator)
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/dataset_olist'

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Step 1: Assemble feature columns into one vector
assembler = VectorAssembler(
    inputCols=['delivery_days', 'total_price', 'freight_value',
               'payment_installments', 'item_count'],
    outputCol='features'
)

# Step 2: Define the Random Forest classifier
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    seed=42
)

# Step 3: Chain into a pipeline
pipeline = Pipeline(stages=[assembler, rf])

# Step 4: Define the hyperparameter grid to search
# Grid Search will try ALL combinations:
# 3 numTrees x 3 maxDepth x 2 minInstancesPerNode = 18 combinations
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees,             [50, 100, 150])  \
    .addGrid(rf.maxDepth,             [4,  6,   8])    \
    .addGrid(rf.minInstancesPerNode,  [1,  5])         \
    .build()

# Step 5: Define the evaluator (optimise for F1 score)
evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
)

# Step 6: Create CrossValidator
# numFolds=3 means each of the 18 combinations is trained and
# evaluated 3 times on different folds -> 54 total model fits
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print('Grid Search setup complete.')
print(f'Total parameter combinations : {len(param_grid)}')
print(f'Cross-validation folds       : 3')
print(f'Total model fits             : {len(param_grid) * 3}')
print()
print('Parameters being searched:')
print('  numTrees            : 50, 100, 150')
print('  maxDepth            : 4, 6, 8')
print('  minInstancesPerNode : 1, 5')

In [0]:
# RUN GRID SEARCH (fit CrossValidator on training data)

print('Running Grid Search with 3-fold Cross-Validation...')
print('This may take several minutes on Community Edition.')
print()

cv_model = cv.fit(train_df)   # Grid Search happens here

print('Grid Search complete!')
print()

# Extract the best pipeline model found by CV
best_model = cv_model.bestModel
best_rf    = best_model.stages[-1]   # Random Forest stage from pipeline

print('=== BEST HYPERPARAMETERS FOUND ===')
print(f'  numTrees            : {best_rf.getNumTrees}')
print(f'  maxDepth            : {best_rf.getMaxDepth()}')
print(f'  minInstancesPerNode : {best_rf.getMinInstancesPerNode()}')
print()

# Show CV scores for every combination tried
print('=== ALL COMBINATIONS TRIED (sorted by F1, best first) ===')
import pandas as pd

results = []
for params, score in zip(param_grid, cv_model.avgMetrics):
    results.append({
        'numTrees'           : params[rf.numTrees],
        'maxDepth'           : params[rf.maxDepth],
        'minInstances'       : params[rf.minInstancesPerNode],
        'avg_CV_F1'          : round(score, 4)
    })

results_df = pd.DataFrame(results).sort_values('avg_CV_F1', ascending=False)
print(results_df.to_string(index=False))

In [0]:
# FINAL EVALUATION ON TEST SET (best model only, run once)

print('=== FINAL TEST SET EVALUATION ===')
print(f'Best params: numTrees={best_rf.getNumTrees}, '
      f'maxDepth={best_rf.getMaxDepth()}, '
      f'minInstancesPerNode={best_rf.getMinInstancesPerNode()}')
print()

test_predictions = best_model.transform(test_df)

# Calculate all metrics
accuracy  = evaluator.setMetricName('accuracy').evaluate(test_predictions)
precision = evaluator.setMetricName('weightedPrecision').evaluate(test_predictions)
recall    = evaluator.setMetricName('weightedRecall').evaluate(test_predictions)
f1        = evaluator.setMetricName('f1').evaluate(test_predictions)

print('================================================')
print('        FINAL MODEL EVALUATION RESULTS         ')
print('================================================')
print(f'  numTrees            : {best_rf.getNumTrees}')
print(f'  maxDepth            : {best_rf.getMaxDepth()}')
print(f'  minInstancesPerNode : {best_rf.getMinInstancesPerNode()}')
print('  ----------------------------------------')
print(f'  Accuracy            : {accuracy:.4f}  ({accuracy*100:.1f}%)')
print(f'  Precision           : {precision:.4f}')
print(f'  Recall              : {recall:.4f}')
print(f'  F1 Score            : {f1:.4f}')
print('================================================')
print()
print('Sample predictions:')
test_predictions.select(
    'delivery_days', 'total_price', 'label', 'prediction'
).show(8)

In [0]:
# FEATURE IMPORTANCE

import pandas as pd

feature_names = ['delivery_days', 'total_price', 'freight_value',
                 'payment_installments', 'item_count']
importances   = best_rf.featureImportances.toArray()

imp_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('=== FEATURE IMPORTANCE (Best Model) ===')
print(imp_df.to_string(index=False))
print()
print('KEY FINDING: delivery_days is the strongest predictor.')
print('Late delivery is the #1 cause of negative reviews on Olist.')

spark_imp = spark.createDataFrame(imp_df)
display(spark_imp)

Databricks visualization. Run in Databricks to view.